<a id="top"></a>

# Demo: three outcomes, including a hallucinated call

```yaml
title: "Demo: three outcomes, including a hallucinated call"
keywords: three outcomes, hallucinated tool call, validation, application validates, tool_choice, haiku contrast, non-determinism, teacher demo
estimated duration: 13
```

> **Lesson:** L07. Teacher demo — Demo 4 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Model behavior **varies**: if Sonnet is too well-behaved on P3, re-run on **Haiku 4.5**, or hand-edit the arguments (last cell) to force the validation path. Clear outputs before committing.
>
> **Why the raw Anthropic SDK here, not `potato_llm`?** The course's `potato_llm` seam is **text-only** — its `Message` carries a string and cannot represent `tool_use` / `tool_result` blocks. L07 is *about* those blocks, so these notebooks call the raw SDK directly. The API key still loads through the config seam (`require_anthropic_key`); we never hard-code it. This is the one lesson that legitimately reaches under the seam.
>
> The point to land: handing the model a tool has **three observable outcomes** — calls it, skips it, or calls it with bad arguments — and the **application**, not the model, catches the bad case. The schema is a contract about shape, not a guarantee about behavior.

## Contents

- [1. Setup — the tool, plus a visible validator](#1-setup--the-tool-plus-a-visible-validator)
- [2. Outcome one — a clean tool call (P1)](#2-outcome-one--a-clean-tool-call-p1)
- [3. Outcome two — the model answers without the tool (P2)](#3-outcome-two--the-model-answers-without-the-tool-p2)
- [4. Outcome three — a malformed / hallucinated call (P3)](#4-outcome-three--a-malformed--hallucinated-call-p3)
- [5. Fallback — force the validation path by hand](#5-fallback--force-the-validation-path-by-hand)
- [6. Takeaways](#6-takeaways)

## 1. Setup — the tool, plus a visible validator

The same `calculator`, plus a `dispatch` helper whose validation step is **visible**: a bad call is *rejected*, not crashed-on. Two clients ready — Sonnet 4.6 (anchor) and Haiku 4.5 (the smaller-model contrast).

In [ ]:
import anthropic

from fluffy_potato_curriculum.common.config import require_anthropic_key

client = anthropic.Anthropic(api_key=require_anthropic_key())
SONNET = "claude-sonnet-4-6"  # course anchor
HAIKU = "claude-haiku-4-5"  # smaller model — fumbles tool selection / args more often


# The ONE tool that carries all four demos: a deterministic calculator.
# L07 is deliberately single-tool, single-round-trip (multi-tool is L08, the
# agent loop is L10). Resist adding a second tool.
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression and return the result as a string.

    Restricted to digits and the operators + - * / ( ) . and whitespace so a
    hallucinated expression cannot run arbitrary code. Raises ValueError on
    anything else — that rejection is the whole point of Demo 4."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the character whitelist above blocks names,
    # attributes, and calls. Never eval untrusted input without such a guard.
    return str(eval(expression))


# The tool DEFINITION: name + natural-language description + JSON-Schema input.
# This is all the model ever sees — never the Python function itself.
CALCULATOR_TOOL: dict[str, object] = {
    "name": "calculator",
    "description": (
        "Evaluate a basic arithmetic expression (the four operators and "
        "parentheses) and return the exact numeric result. Use this whenever "
        "the user asks for the result of a calculation."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The arithmetic expression to evaluate, e.g. '18374 * 92431'.",
            }
        },
        "required": ["expression"],
    },
}


def dispatch(tool_use) -> str:
    """Validate a tool_use block and run the tool, or return a loud rejection string."""
    if tool_use.name != "calculator":
        return f"REJECTED: no such tool {tool_use.name!r} (the model invented it)"
    args = tool_use.input
    if "expression" not in args:
        return f"REJECTED: missing required argument 'expression' (got {args!r})"
    try:
        return f"OK: calculator({args['expression']!r}) = {calculator(args['expression'])}"
    except ValueError as exc:
        return f"REJECTED: {exc}"


def first_block(model: str, prompt: str):
    """Return the response so the class can read its blocks."""
    return client.messages.create(
        model=model,
        max_tokens=400,
        tools=[CALCULATOR_TOOL],
        messages=[{"role": "user", "content": prompt}],
    )


print("setup ready")

[↑ Back to top](#top)

## 2. Outcome one — a clean tool call (P1)

Arithmetic the model should defer to the tool. A `tool_use` block, valid args, a real result.

In [ ]:
resp = first_block(SONNET, "What is 47,219 multiplied by 8,803?")
print("blocks:", [b.type for b in resp.content])
for b in resp.content:
    if b.type == "tool_use":
        print("dispatch ->", dispatch(b))

[↑ Back to top](#top)

## 3. Outcome two — the model answers without the tool (P2)

A trivial sum the model already owns. Usually it answers directly — only a `text` block, no `tool_use`. A registered tool is an **option**, never an obligation (adding a tool to a task the model already owns is wasted overhead — a forward-link to [L08](L0702_lecture.md)).

In [ ]:
resp = first_block(SONNET, "What is 2 + 2?")
print("blocks:", [b.type for b in resp.content])
called = any(b.type == "tool_use" for b in resp.content)
print("did the model call the tool?", called)

[↑ Back to top](#top)

## 4. Outcome three — a malformed / hallucinated call (P3)

A prompt that nudges the model toward a bad argument (a non-numeric token in the expression). The model may emit a `tool_use` block whose arguments are malformed — and the **application's validation catches it**. If Sonnet sanitizes cleanly, re-run on Haiku.

In [ ]:
P3 = "Use the calculator to work out the average of these three: 12, 19, and twenty."

for model in (SONNET, HAIKU):
    resp = first_block(model, P3)
    print(f"=== {model} ===")
    print("  blocks:", [b.type for b in resp.content])
    for b in resp.content:
        if b.type == "tool_use":
            print("  proposed args:", b.input)
            print("  dispatch ->", dispatch(b))

[↑ Back to top](#top)

## 5. Fallback — force the validation path by hand

If neither model fumbles on the day, hand-edit a `tool_use` block to a bad value and feed it to `dispatch`. The teaching point is the application's **response** to a bad call, which you can show regardless of whether the model cooperated.

In [ ]:
from types import SimpleNamespace

# A hand-built 'tool_use' that mimics three different hallucinations.
bad_calls = [
    SimpleNamespace(name="calculator", input={"expression": "twenty + 1"}),  # non-numeric
    SimpleNamespace(name="calculator", input={}),  # missing arg
    SimpleNamespace(name="wikipedia", input={"query": "geese"}),  # invented tool
]
for call in bad_calls:
    print(dispatch(call))

[↑ Back to top](#top)

## 6. Takeaways

- The model can **hallucinate** a tool call: wrong types, missing required args, invented args, even a tool name that doesn't exist. The schema stopped **none** of it at generation time — **the application validates; the model proposes.**
- Showing one hallucination is worth ten clean runs — the L07 analogue of L06's tag-violation moment.
- Tool calls are **not deterministic**: same prompt + schema can call, skip, or pass different arguments across runs. That is *why* validation is not optional.
- `tool_choice` (auto / any / none / specific) can *bias* the decision, but forcing a call still does not guarantee well-formed arguments. Designing the error *response* is [L08](L0702_lecture.md)'s job, not L07's.

[↑ Back to top](#top)